In [1]:
!pip install pyspark

In [23]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import count, desc
from pyspark.sql.functions import sum as spark_sum

In [2]:
spark = SparkSession.builder \
    .appName("Test") \
    .master("local[*]") \
    .getOrCreate()

print("Spark works!")

Spark works!


In [7]:
from google.colab import files
uploaded = files.upload()

Saving actor.csv to actor.csv
Saving address.csv to address.csv
Saving category.csv to category.csv
Saving city.csv to city.csv
Saving customer.csv to customer.csv
Saving film.csv to film.csv
Saving film_actor.csv to film_actor.csv
Saving film_category.csv to film_category.csv
Saving inventory.csv to inventory.csv
Saving payment.csv to payment.csv
Saving rental.csv to rental.csv


In [8]:
film = spark.read.csv("film.csv", header=True, inferSchema=True)
category = spark.read.csv("category.csv", header=True, inferSchema=True)
film_category = spark.read.csv("film_category.csv", header=True, inferSchema=True)
actor = spark.read.csv("actor.csv", header=True, inferSchema=True)
film_actor = spark.read.csv("film_actor.csv", header=True, inferSchema=True)
inventory = spark.read.csv("inventory.csv", header=True, inferSchema=True)
rental = spark.read.csv("rental.csv", header=True, inferSchema=True)
payment = spark.read.csv("payment.csv", header=True, inferSchema=True)
customer = spark.read.csv("customer.csv", header=True, inferSchema=True)
address = spark.read.csv("address.csv", header=True, inferSchema=True)
city = spark.read.csv("city.csv", header=True, inferSchema=True)

In [9]:
film.show(5)

+-------+----------------+--------------------+------------+-----------+--------------------+---------------+-----------+------+----------------+------+--------------------+--------------------+--------------------+
|film_id|           title|         description|release_year|language_id|original_language_id|rental_duration|rental_rate|length|replacement_cost|rating|         last_update|    special_features|            fulltext|
+-------+----------------+--------------------+------------+-----------+--------------------+---------------+-----------+------+----------------+------+--------------------+--------------------+--------------------+
|      1|ACADEMY DINOSAUR|A Epic Drama of a...|        2006|          1|                NULL|              6|       0.99|    86|           20.99|    PG|2022-09-10 16:46:...|"{""Deleted Scenes""|""Behind the Scen...|
|      2|  ACE GOLDFINGER|A Astounding Epis...|        2006|          1|                NULL|              3|       4.99|    48|        

In [10]:
film.write.mode("overwrite").parquet("film_parquet")
category.write.mode("overwrite").parquet("category_parquet")
film_category.write.mode("overwrite").parquet("film_category_parquet")
actor.write.mode("overwrite").parquet("actor_parquet")
film_actor.write.mode("overwrite").parquet("film_actor_parquet")
inventory.write.mode("overwrite").parquet("inventory_parquet")
rental.write.mode("overwrite").parquet("rental_parquet")
payment.write.mode("overwrite").parquet("payment_parquet")
customer.write.mode("overwrite").parquet("customer_parquet")
address.write.mode("overwrite").parquet("address_parquet")
city.write.mode("overwrite").parquet("city_parquet")

In [11]:
film = spark.read.parquet("film_parquet")
category = spark.read.parquet("category_parquet")
film_category = spark.read.parquet("film_category_parquet")
actor = spark.read.parquet("actor_parquet")
film_actor = spark.read.parquet("film_actor_parquet")
inventory = spark.read.parquet("inventory_parquet")
rental = spark.read.parquet("rental_parquet")
payment = spark.read.parquet("payment_parquet")
customer = spark.read.parquet("customer_parquet")
address = spark.read.parquet("address_parquet")
city = spark.read.parquet("city_parquet")

Output the number of movies in each category, sorted in descending order.

In [13]:
film_with_category = film_category.join(
    category,
    on = "category_id",
    how = "inner"
)

film_with_category.show(5)

+-----------+-------+-------------------+-----------+-------------------+
|category_id|film_id|        last_update|       name|        last_update|
+-----------+-------+-------------------+-----------+-------------------+
|          6|      1|2022-02-15 10:07:09|Documentary|2022-02-15 09:46:27|
|         11|      2|2022-02-15 10:07:09|     Horror|2022-02-15 09:46:27|
|          6|      3|2022-02-15 10:07:09|Documentary|2022-02-15 09:46:27|
|         11|      4|2022-02-15 10:07:09|     Horror|2022-02-15 09:46:27|
|          8|      5|2022-02-15 10:07:09|     Family|2022-02-15 09:46:27|
+-----------+-------+-------------------+-----------+-------------------+
only showing top 5 rows


In [18]:
movies_per_category = film_with_category.groupBy("name").agg(count("film_id").alias("movie_count"))
result = movies_per_category.orderBy(desc("movie_count"))
result.show()

+-----------+-----------+
|       name|movie_count|
+-----------+-----------+
|     Sports|         74|
|    Foreign|         73|
|     Family|         69|
|Documentary|         68|
|  Animation|         66|
|     Action|         64|
|        New|         63|
|      Drama|         62|
|      Games|         61|
|     Sci-Fi|         61|
|   Children|         60|
|     Comedy|         58|
|     Travel|         57|
|   Classics|         57|
|     Horror|         56|
|      Music|         51|
+-----------+-----------+



Output the 10 actors whose movies rented the most, sorted in descending order.

In [20]:
actor_rentals = actor \
.join(film_actor, on="actor_id", how = 'inner') \
.join(inventory, on = "film_id", how="inner") \
.join(rental, on="inventory_id", how= "inner")

actor_rentals.show(5)

+------------+-------+--------+----------+---------+-------------------+-------------------+--------+-------------------+---------+-------------------+-----------+-------------------+--------+-------------------+
|inventory_id|film_id|actor_id|first_name|last_name|        last_update|        last_update|store_id|        last_update|rental_id|        rental_date|customer_id|        return_date|staff_id|        last_update|
+------------+-------+--------+----------+---------+-------------------+-------------------+--------+-------------------+---------+-------------------+-----------+-------------------+--------+-------------------+
|           8|      1|       1|  PENELOPE|  GUINESS|2022-02-15 09:34:33|2022-02-15 10:05:03|       2|2022-02-15 10:09:17|    12651|2022-08-18 17:36:16|         34|2022-08-22 21:01:16|       1|2022-02-16 02:30:53|
|           8|      1|       1|  PENELOPE|  GUINESS|2022-02-15 09:34:33|2022-02-15 10:05:03|       2|2022-02-15 10:09:17|    10141|2022-07-31 21:08:

In [21]:
top_actors = actor_rentals \
  .groupBy("actor_id", "first_name", "last_name") \
  .agg(count("rental_id").alias("rental_count")) \
  .orderBy(desc('rental_count')) \
  .limit(10)

top_actors.show()

+--------+----------+-----------+------------+
|actor_id|first_name|  last_name|rental_count|
+--------+----------+-----------+------------+
|     107|      GINA|  DEGENERES|         753|
|     181|   MATTHEW|     CARREY|         678|
|     198|      MARY|     KEITEL|         674|
|     144|    ANGELA|WITHERSPOON|         654|
|     102|    WALTER|       TORN|         640|
|      60|     HENRY|      BERRY|         612|
|     150|     JAYNE|      NOLTE|         611|
|      37|       VAL|     BOLGER|         605|
|      23|    SANDRA|     KILMER|         604|
|      90|      SEAN|    GUINESS|         599|
+--------+----------+-----------+------------+



Output the category of movies on which the most money was spent.

In [25]:
category_revenue = payment \
    .join(rental, on="rental_id", how="inner") \
    .join(inventory, on="inventory_id", how="inner") \
    .join(film_category, on="film_id", how="inner") \
    .join(category, on="category_id", how="inner")


revenue_per_category = category_revenue \
  .groupBy('name') \
  .agg(spark_sum("amount").alias("total_revenue")) \
  .orderBy(desc("total_revenue"))

In [26]:
top_category = revenue_per_category.limit(1)

top_category.show()

+------+-----------------+
|  name|    total_revenue|
+------+-----------------+
|Sports|5314.209999999847|
+------+-----------------+



Output the names of movies that are not in the inventory.

In [27]:
movies_not_in_inventory = film.join(
    inventory,
    on="film_id",
    how="left_anti"
)

movies_not_in_inventory.select("title").show()

+--------------------+
|               title|
+--------------------+
|      ALICE FANTASIA|
|         APOLLO TEEN|
|      ARGONAUTS TOWN|
|       ARK RIDGEMONT|
|ARSENIC INDEPENDENCE|
|   BOONDOCK BALLROOM|
|       BUTCH PANTHER|
|       CATCH AMISTAD|
| CHINATOWN GLADIATOR|
|      CHOCOLATE DUCK|
|COMMANDMENTS EXPRESS|
|    CROSSING DIVORCE|
|     CROWDS TELEMARK|
|    CRYSTAL BREAKING|
|          DAZED PUNK|
|DELIVERANCE MULHO...|
|   FIREHOUSE VIETNAM|
|       FLOATS GARDEN|
|FRANKENSTEIN STRA...|
|  GLADIATOR WESTWARD|
+--------------------+
only showing top 20 rows
